# Install the required packages
!pip install -r requirements.txt

In [ ]:
!bash prereqs/deploy_OSS.sh

In [ ]:
wait_time = 120
print(f"Waiting {wait_time} seconds for OpenSearch Serverless collection to be fully operational...")
time.sleep(wait_time)

OPENSEARCH_HOST: qjfd2k8ezfmfhsbvh6g7.us-west-2.aoss.amazonaws.com
OPENSEARCH_COLLECTION_ID: qjfd2k8ezfmfhsbvh6g7

In [ ]:
# Import Required Libraries
import os
from strands import Agent, tool
from strands.models import bedrock
from strands_tools import mem0_memory
import time

from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-2")
bedrock.DEFAULT_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0" 
#Optional: Set a default model for Bedrock

In [ ]:
os.environ["OPENSEARCH_HOST"] = "qjfd2k8ezfmfhsbvh6g7.us-west-2.aoss.amazonaws.com"
os.environ["MEM0_API_KEY"] = "

In [ ]:
# Option 1: Opensearch Serverless
from dotenv import load_dotenv
load_dotenv() # take Opensearch environment variables

In [ ]:
# Define a focused system prompt for memory operations
SYSTEM_PROMPT = """You are a helpful personal assistant for a user. Your task is to assist the user by providing personalized responses based on their history. 

Capabilities:
- You can store information using the mem0_memory tool (action="store").
- You can retrieve relevant memories using the mem0_memory tool (action="retrieve").
- You can use the websearch tool to find information on the web.

Key Rules:
- Be conversational and natural in your responses.
- Always retrieve memories before responding to the user and use them to inform your response.
- Store any new user information and user preferences in mem0_memory.
- Only share relevant information.
- Politely indicate when you don't have the information.
"""

In [ ]:
@tool
def websearch(
    keywords: str,
    region: str = "kr-kr",
    max_results: int | None = None,
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

In [ ]:
# Create an agent with memory, websearch tool
USER_ID = "user123" # Replace with actual user ID

memory_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    tools=[mem0_memory, websearch],
)

In [16]:
# Store initial memories to demonstrate retrieval
memory_agent.tool.mem0_memory(
    action="store", content=f"The user's name is {USER_ID}.", user_id=USER_ID
)

╭──────────────────────────────────────────── Memory for user user123 ────────────────────────────────────────────╮
│ The user's name is user123.                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ object of type 'NoneType' has no len()                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_333860405',
 'status': 'error',
 'content': [{'text': "Error: object of type 'NoneType' has no len()"}]}

In [17]:
memory_agent.tool.mem0_memory(
    action="store", 
    content="I like to drink tea more than coffee.", 
    user_id=USER_ID
)

╭──────────────────────────────────────────── Memory for user user123 ────────────────────────────────────────────╮
│ I like to drink tea more than coffee.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ object of type 'NoneType' has no len()                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_943505722',
 'status': 'error',
 'content': [{'text': 'The tool result was too large!'}]}

In [18]:
# Retrieve memories
retrieved_memories = memory_agent.tool.mem0_memory(
    action="retrieve", query="What is the user's name?", user_id=USER_ID
)
print("Retrieved Memories:", retrieved_memories)

HTTP error occurred: Client error '400 Bad Request' for url 'https://api.mem0.ai/v2/memories/search/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ {"error":"Filters are required and cannot be empty. Please refer to                                             │
│ https://docs.mem0.ai/api-reference/memory/search-memories"}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Retrieved Memories: {'toolUseId': 'tooluse_mem0_memory_108453594', 'status': 'error', 'content': [{'text': 'The tool result was too large!'}]}


In [19]:
# Retrieve memories about preferences
memory_agent.tool.mem0_memory(
    action="retrieve",
    query="What are the my drink preferences?",
    user_id=USER_ID
)

HTTP error occurred: Client error '400 Bad Request' for url 'https://api.mem0.ai/v2/memories/search/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ {"error":"Filters are required and cannot be empty. Please refer to                                             │
│ https://docs.mem0.ai/api-reference/memory/search-memories"}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_899701460',
 'status': 'error',
 'content': [{'text': 'The tool result was too large!'}]}

In [20]:
# Ask the agent a question
response = memory_agent("What are the events happening in the New York today?")
print(response)

I'll search for current events happening in New York today.
Tool #1: mem0_memory

Tool #2: websearch


HTTP error occurred: Client error '400 Bad Request' for url 'https://api.mem0.ai/v2/memories/search/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ {"error":"Filters are required and cannot be empty. Please refer to                                             │
│ https://docs.mem0.ai/api-reference/memory/search-memories"}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Based on my search, here are some resources for finding events happening in New York today:

**Top Event Resources:**

1. **Eventbrite** - Browse various activities and events happening today across New York, including categories like:
   - Music & concerts
   - Film & media
   - Performing & visual arts
   - Food & drink
   - Sports & fitness
   - Comedy shows
   - And much more

2. **DoNYC.com** - Daily updated listings of:
   - Concerts
   - Comedy shows
   - Nightlife events
   - Free events
   - With ticket information and times

3. **Happening NYC** - Curated events specifically for today in New York City

4. **Time Out New York** - Features:
   - Live shows
   - Art exhibitions
   - Kick-ass fun activities
   - Cultural events around the city

5. **I Love NY Events Calendar** - State-wide events including:
   - Festivals and fairs
   - Live music
   - Seasonal happenings

To get the most current and specific events for today, I'd recommend visiting these websites directly, as th

In [21]:
# List all stored memories
print("All Stored Memories:")
all_memories = memory_agent.tool.mem0_memory(
    action="list", user_id=USER_ID
)

HTTP error occurred: Client error '400 Bad Request' for url 'https://api.mem0.ai/v2/memories/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


All Stored Memories:


╭─────────────────────────────────────────── ❌ Memory Operation Error ───────────────────────────────────────────╮
│ {"error":"Filters are required and cannot be empty. Please refer to                                             │
│ https://docs.mem0.ai/api-reference/memory/get-memories"}                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯